XML Parsing

In [70]:
import xml.etree.ElementTree as ET
from datetime import datetime
import numpy as np

Load XML

In [71]:
tree = ET.parse("Instance1.xml")
root = tree.getroot()

Read Planning Period (p)

In [72]:
start = datetime.fromisoformat(root.find("StartDate").text)
end = datetime.fromisoformat(root.find("EndDate").text)
P = (end - start).days + 1
print("Days:", P)

Days: 14


Read shift typed

In [73]:
shift_types = []
shift_dict = {}

for i, sh in enumerate(root.find("ShiftTypes")):
    shift_id = sh.attrib["ID"]   # e.g. "D"
    shift_types.append(shift_id)
    shift_dict[shift_id] = i

NUM_SHIFTS = len(shift_types)
print("Shifts:", shift_types)
print("Shift", shift_dict)

Shifts: ['D']
Shift {'D': 0}


Read Employees

In [74]:
employees = []
employee_contracts = {}

for i, emp in enumerate(root.find("Employees")):
    emp_id = emp.attrib["ID"]
    key=i
    employees.append(emp_id)
    employee_contracts[emp_id] = [
        c.text for c in emp.findall("ContractID")
    ]

S = len(employees)
print("Nurses:", employees)
print(employee_contracts)

emp_dict = {}
for l, ip in enumerate(employees):
    print(f"{l}. Contract of {ip} is {employee_contracts[ip]}")

    emp_dict[ip] = l

print(emp_dict)



Nurses: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']
{'A': ['All', 'A'], 'B': ['All', 'B'], 'C': ['All', 'C'], 'D': ['All', 'D'], 'E': ['All', 'E'], 'F': ['All', 'F'], 'G': ['All', 'G'], 'H': ['All', 'H']}
0. Contract of A is ['All', 'A']
1. Contract of B is ['All', 'B']
2. Contract of C is ['All', 'C']
3. Contract of D is ['All', 'D']
4. Contract of E is ['All', 'E']
5. Contract of F is ['All', 'F']
6. Contract of G is ['All', 'G']
7. Contract of H is ['All', 'H']
{'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'G': 6, 'H': 7}


Build demand matrix from <CoverRequirements> 

We build:
DEMAND[day][shift] = required nurses

In [75]:
DEMAND = np.zeros((P, NUM_SHIFTS), dtype=int)

for cover_day in root.find("CoverRequirements"):
    day = int(cover_day.find("Day").text)
    cover = cover_day.find("Cover")
    sh = cover.find("Shift").text
    min_required = int(cover.find("Min").text)

    DEMAND[day, shift_index[sh]] = min_required
for i,_ in enumerate(DEMAND):
    print(f"{i}: {DEMAND[i]} or {_} ")

for k in DEMAND:
    print(f": {DEMAND[k]} ")

0: [5] or [5] 
1: [7] or [7] 
2: [6] or [6] 
3: [4] or [4] 
4: [5] or [5] 
5: [5] or [5] 
6: [5] or [5] 
7: [6] or [6] 
8: [7] or [7] 
9: [4] or [4] 
10: [2] or [2] 
11: [5] or [5] 
12: [6] or [6] 
13: [4] or [4] 
: [[5]] 
: [[6]] 
: [[5]] 
: [[5]] 
: [[5]] 
: [[5]] 
: [[5]] 
: [[5]] 
: [[6]] 
: [[5]] 
: [[6]] 
: [[5]] 
: [[5]] 
: [[5]] 


Fixed Assignments

In [76]:
fixed_assignments = []

for emp in root.find("FixedAssignments"):
    emp_id = emp.find("EmployeeID").text
    assign = emp.find("Assign")
    shift = assign.find("Shift").text
    day = int(assign.find("Day").text)

    fixed_assignments.append((day, shift, emp_id))
w=len(fixed_assignments)
print(fixed_assignments)
print(w)

[(0, '-', 'A'), (5, '-', 'B'), (8, '-', 'C'), (2, '-', 'D'), (9, '-', 'E'), (5, '-', 'F'), (1, '-', 'G'), (7, '-', 'H')]
8


Genetic Algorithm

Chromosome Representation

# schedule[nurse][day][shift] ∈ {0,1}
def random_schedule():
    sched = np.zeros((S, P, NUM_SHIFTS), dtype=int)

    for n in range(S):
        for d in range(P):
            if np.random.rand() < 0.6:
                sh = np.random.randint(NUM_SHIFTS)
                sched[n, d, sh] = 1
    return sched

Fitness Function

In [77]:
def random_schedule():
    sched = np.zeros((S, P, NUM_SHIFTS), dtype=int)

    for i in range(w):

        day, shift, nurse_id = fixed_assignments(i)
        nurse = emp_dict[nurse_id]

        if shift=='-':
            shift = np.random.randint(NUM_SHIFTS)
            sched[nurse, day, shift] = 1
        else:
            shift = shift_dict[shift]
            sched[nurse, day, shift] = 1
                
    for n in range(S):
        for d in range(P):
            if np.random.rand() < 0.6:
                sh = np.random.randint(NUM_SHIFTS)
                sched[n, d, sh] = 1
    return sched

In [78]:
HARD_PENALTY = 10_000
SOFT_PENALTY = 10

def fitness(schedule):
    penalty = 0

    # ---- Hard: one shift per nurse per day
    penalty += np.sum(schedule.sum(axis=2) > 1) * HARD_PENALTY

    # ---- Hard: cover demand
    for d in range(P):
        for sh in range(NUM_SHIFTS):
            assigned = schedule[:, d, sh].sum()
            if assigned < DEMAND[d, sh]:
                penalty += (DEMAND[d, sh] - assigned) * HARD_PENALTY

    # ---- Hard: fixed assignments
    for emp_id, day, sh in fixed_assignments:
        n = employees.index(emp_id)
        if sh == "-":
            if schedule[n, day].sum() > 0:
                penalty += HARD_PENALTY
        else:
            s_idx = shift_index[sh]
            if schedule[n, day, s_idx] == 0:
                penalty += HARD_PENALTY

    # ---- Hard: max 5 consecutive working days
    for n in range(S):
        consec = 0
        for d in range(P):
            if schedule[n, d].sum() > 0:
                consec += 1
                if consec > 5:
                    penalty += HARD_PENALTY
            else:
                consec = 0

    return -penalty

GA Loop

In [79]:
POP = 60
GENS = 300

population = [random_schedule() for _ in range(POP)]

#print("random population", population)

for gen in range(GENS):
    scores = [fitness(ind) for ind in population]

    new_pop = []
    elite = population[np.argmax(scores)]
    new_pop.append(elite.copy())

    while len(new_pop) < POP:
        p1 = population[np.argmax(np.random.choice(scores, 3))]
        p2 = population[np.argmax(np.random.choice(scores, 3))]
        child = p1.copy()
        mask = np.random.rand(S, P, NUM_SHIFTS) < 0.5
        child[mask] = p2[mask]
        new_pop.append(child)

    population = new_pop[:POP]

#print("selected some", population)

TypeError: 'list' object is not callable

Output

In [ ]:
best = population[np.argmax([fitness(ind) for ind in population])]

print("\nFinal Schedule:")
for d in range(P):
    for sh, name in enumerate(shift_types):
        row = [
            "Y" if best[n, d, sh] else "N"
            for n in range(S)
        ]
        print(f"Day {d}, Shift {name}: {row}")


Final Schedule:
Day 0, Shift D: ['N', 'Y', 'Y', 'Y', 'Y', 'Y', 'Y', 'Y']
Day 1, Shift D: ['Y', 'Y', 'Y', 'Y', 'Y', 'Y', 'Y', 'N']
Day 2, Shift D: ['Y', 'Y', 'Y', 'Y', 'N', 'N', 'Y', 'Y']
Day 3, Shift D: ['N', 'Y', 'N', 'Y', 'Y', 'Y', 'N', 'Y']
Day 4, Shift D: ['N', 'N', 'Y', 'N', 'Y', 'N', 'Y', 'Y']
Day 5, Shift D: ['N', 'Y', 'Y', 'Y', 'Y', 'N', 'Y', 'Y']
Day 6, Shift D: ['N', 'N', 'Y', 'Y', 'Y', 'Y', 'Y', 'N']
Day 7, Shift D: ['Y', 'N', 'N', 'Y', 'Y', 'N', 'Y', 'Y']
Day 8, Shift D: ['Y', 'N', 'Y', 'Y', 'N', 'Y', 'Y', 'Y']
Day 9, Shift D: ['N', 'Y', 'Y', 'N', 'Y', 'N', 'Y', 'N']
Day 10, Shift D: ['Y', 'N', 'N', 'N', 'N', 'Y', 'N', 'N']
Day 11, Shift D: ['N', 'Y', 'Y', 'N', 'Y', 'Y', 'Y', 'N']
Day 12, Shift D: ['Y', 'N', 'Y', 'N', 'Y', 'Y', 'N', 'Y']
Day 13, Shift D: ['Y', 'Y', 'N', 'N', 'Y', 'N', 'Y', 'Y']
